In [9]:
from pathlib import Path
import sys
import warnings
import re # re stands for Regular Expressions. It is used for text cleaning.
import html # Used to decode HTML entities.Very useful for Youtube comments, as they often contain HTML entities.

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" # Comments Sentiment Analysis/data/
RAW_DATA_PATH = DATA_DIR / "comments.csv"
CLEANED_DATA_PATH = DATA_DIR / "cleaned_comments.csv" # Output file location after preprocessing.
REQUIRED_COLUMNS = ["CommentText", "Sentiment"] # The only columns needed for analysis
RANDOM_STATE = 42

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")

from utils import load_raw_data

#Check for langdetect
try: #Attempts to import the language detection library.
    from langdetect import detect, DetectorFactory # It identifies the language of a piece of text. #DetectorFactory is used to make the language detection deterministic so the same text always yields the same language prediction.
except ImportError as exc: #If it isn't installed,raises a more helpful error:
    raise ImportError(
        "Install langdetect before running this notebook: pip install langdetect"
    ) from exc
# nltk-Natural Language Toolkit)-A library for natural language processing tasks like tokenization, stemming, and lemmatization.
import nltk 
from nltk.corpus import stopwords #Stopwords-Words that usually don't add much meaning.ex-is,the,a,an
from nltk.stem import WordNetLemmatizer # Lemmatizer - Converts words to their base form. ex- running -> run, better -> good

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


True

 ------------------------------------------------------------------
 1. Load raw data and keep the fields used for modeling
 ------------------------------------------------------------------


In [10]:
df = load_raw_data(RAW_DATA_PATH) # See cell 1 for reference

missing_columns = set(REQUIRED_COLUMNS) - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = df[REQUIRED_COLUMNS].copy()
df["CommentText"] = df["CommentText"].astype("string").str.strip() #astype("string")-->Converts everything into text,.str.strip()-->Removes spaces.
df["Sentiment"] = df["Sentiment"].astype("string").str.strip()

print("Raw modeling shape:", df.shape)
print("Missing values before cleanup:")
print(df.isnull().sum())

before = len(df)
df = df.dropna(subset=REQUIRED_COLUMNS)
df = df[df["CommentText"].str.len() > 0].reset_index(drop=True)

print(f"Dropped {before - len(df)} rows with missing/blank required values")
print("Shape after basic cleanup:", df.shape)
print("Sentiment distribution:")
print(df["Sentiment"].value_counts())


Raw modeling shape: (103482, 2)
Missing values before cleanup:
CommentText    0
Sentiment      0
dtype: int64
Dropped 0 rows with missing/blank required values
Shape after basic cleanup: (103482, 2)
Sentiment distribution:
Sentiment
Negative    35073
Positive    34308
Neutral     34101
Name: count, dtype: Int64


 ------------------------------------------------------------------
 2. Remove duplicate comments
 ------------------------------------------------------------------


In [11]:
before = len(df)
duplicate_key = df["CommentText"].str.lower()
df = df.loc[~duplicate_key.duplicated()].reset_index(drop=True)

print(f"Dropped {before - len(df)} duplicate comments -> shape {df.shape}")
print("Sentiment distribution after de-duplication:")
print(df["Sentiment"].value_counts())


Dropped 2584 duplicate comments -> shape (100898, 2)
Sentiment distribution after de-duplication:
Sentiment
Negative    34752
Neutral     33259
Positive    32887
Name: count, dtype: Int64


 ------------------------------------------------------------------
 3. Detect comment language
 ------------------------------------------------------------------


In [12]:
DetectorFactory.seed = RANDOM_STATE


def detect_language(text: str) -> str:
    text = str(text).strip()
    if len(text) < 3:
        return "unknown"
    try:
        return detect(text)
    except Exception:
        return "unknown"


df["Language"] = df["CommentText"].apply(detect_language)

print("Detected language counts:")
print(df["Language"].value_counts().head(10))
df[["CommentText", "Language"]].head()


Detected language counts:
Language
en         90617
tl           778
af           766
de           695
so           689
fr           592
unknown      587
nl           520
id           479
no           465
Name: count, dtype: int64


,CommentText,Language
0,Anyone know what movie this is?,en
1,The fact they're holding each other back while...,en
2,waiting next video will be?,en
3,Thanks for the great video. I don't understand...,en
4,Good person helping good people. This is how i...,en


In [17]:
before = len(df)
df_en = df.loc[df["Language"].eq("en")].reset_index(drop=True)
after = len(df_en)

print(f"Kept {after} / {before} comments ({after / before:.1%}) after English-only filter")
print(f"Dropped {before - after} non-English/unknown comments")
print("Sentiment distribution after language filter:")
print(df_en["Sentiment"].value_counts())


Kept 90477 / 90477 comments (100.0%) after English-only filter
Dropped 0 non-English/unknown comments
Sentiment distribution after language filter:
Sentiment
Negative    32401
Positive    29679
Neutral     28397
Name: count, dtype: Int64


 ------------------------------------------------------------------
 4. Define the text-cleaning function
- lowercase
- remove URLs, HTML tags, @mentions
- preserve negation in contractions such as don't/can't
- remove numbers/punctuation/emoji
- remove stopwords (keeping negations like *no/not/nor*)
- lemmatize tokens
 ------------------------------------------------------------------


In [14]:
stop_words = set(stopwords.words("english")) 
stop_words -= {"no", "not", "nor"}  # keep negations, they carry sentiment . Remove three words from stopwords
lemmatizer = WordNetLemmatizer()  

URL_RE = re.compile(r"https?://\S+|www\.\S+") 
HTML_RE = re.compile(r"<.*?>")
MENTION_RE = re.compile(r"@\w+") 
NON_ALPHA_RE = re.compile(r"[^a-z\s]") 
MULTISPACE_RE = re.compile(r"\s+")

# sub means subsitute 
def clean_text(text: str) -> str:
    text = html.unescape(str(text).lower()) 
    text = re.sub(r"\b(can't|cannot)\b", "can not", text) 
    text = re.sub(r"n't\b", " not", text) 
    text = URL_RE.sub(" ", text) 
    text = HTML_RE.sub(" ", text) 
    text = MENTION_RE.sub(" ", text) 
    text = NON_ALPHA_RE.sub(" ", text) 
    text = MULTISPACE_RE.sub(" ", text).strip() 

    tokens = [] 
    for t in text.split():  
        if t not in stop_words and len(t) > 1: 
            tokens.append(lemmatizer.lemmatize(t))

    return " ".join(tokens)   

In [15]:
df_clean = df_en.copy() 
df_clean["clean_text"] = df_clean["CommentText"].apply(clean_text) 

before = len(df_clean)
df_clean = df_clean[df_clean["clean_text"].str.len() > 0].reset_index(drop=True) 

print(f"Dropped {before - len(df_clean)} rows that became empty after cleaning")
print("Final shape:", df_clean.shape)
print("Final sentiment distribution:")
print(df_clean["Sentiment"].value_counts())

df_clean = df_clean[["CommentText", "Sentiment", "Language", "clean_text"]]
df_clean.to_csv(CLEANED_DATA_PATH, index=False) 
print(f"Saved cleaned dataset to {CLEANED_DATA_PATH}")

df = df_clean


Dropped 140 rows that became empty after cleaning
Final shape: (90477, 4)
Final sentiment distribution:
Sentiment
Negative    32401
Positive    29679
Neutral     28397
Name: count, dtype: Int64
Saved cleaned dataset to /Users/tiyasharma/Desktop/TIYA/AI ML/Project /Comments Sentimental Analysis  copy/data/cleaned_comments.csv


In [16]:
print("Examples:\n")
for i in range(min(5, len(df))):
    print("RAW  :", df["CommentText"].iloc[i][:120])
    print("CLEAN:", df["clean_text"].iloc[i][:120])
    print("-" * 60)


Examples:

RAW  : Anyone know what movie this is?
CLEAN: anyone know movie
------------------------------------------------------------
RAW  : The fact they're holding each other back while equally being most aggressive
CLEAN: fact holding back equally aggressive
------------------------------------------------------------
RAW  : waiting next video will be?
CLEAN: waiting next video
------------------------------------------------------------
RAW  : Thanks for the great video. I don't understand why the DB continues to be accesible through port 8080 when the local mac
CLEAN: thanks great video not understand db continues accesible port local machine connects docker container port not possible 
------------------------------------------------------------
RAW  : Good person helping good people. This is how it is in America with the exception of NY and DC.
CLEAN: good person helping good people america exception ny dc
------------------------------------------------------------
